# Machine Learning Models (Arrivals & Nights)
Modèles : Regression linéaire, Ridge, Decision Tree, SVM (RBF)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import os
import warnings
import sys
sys.path.append('../src')
from autoresearch import AutoResearchEvaluator
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)
auto_eval = AutoResearchEvaluator(output_dir='results/autoresearch_output')

targets = ['Arrivals', 'Nights']
all_results = []

ml_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'SVM (RBF)': SVR(kernel='rbf', C=100, gamma='auto')
}

for target in targets:
    print(f"\n--- Cible : {target} ---")
    df_x_train = pd.read_csv(f'../data/separted/X_train_{target}.csv')
    df_y_train = pd.read_csv(f'../data/separted/y_train_{target}.csv')
    df_x_test = pd.read_csv(f'../data/separted/X_test_{target}.csv')
    df_y_test = pd.read_csv(f'../data/separted/y_test_{target}.csv')

    # Nettoyage et Imputation & Scaling
    cols = ['GDP_Construction_lag1']
    df_x_train = df_x_train.drop(columns=cols, errors='ignore')
    df_x_test = df_x_test.drop(columns=cols, errors='ignore')

    imputer = SimpleImputer(strategy='mean')
    df_x_train_imputed = imputer.fit_transform(df_x_train)
    df_x_test_imputed = imputer.transform(df_x_test)

    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(df_x_train_imputed)
    x_test_scaled = scaler.transform(df_x_test_imputed)

    y_train = df_y_train.values.flatten()
    y_test = df_y_test.values.flatten()

    for name, model in ml_models.items():
        print(f"Training {name} for {target}...")
        model.fit(x_train_scaled, y_train)
        preds = model.predict(x_test_scaled)
        
        r2 = r2_score(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        mape = mean_absolute_percentage_error(y_test, preds)
        
        # AutoResearch evaluation
        res_dict = auto_eval.evaluate_model(target_name=target, model_name=name, y_true=y_test, y_pred=preds, is_walk_forward=False)
        print(f"AutoResearch Insights: {res_dict['Insights']}")
        
        all_results.append({
            'Target': target,
            'Model': name,
            'Type': 'Machine Learning',
            'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape
        })
        
        plt.figure(figsize=(10,5))
        plt.plot(y_test, label='Real', color='black')
        plt.plot(preds, label=f'{name} Prediction', color='blue', linestyle='--')
        plt.title(f'{name} - {target} - Real vs Prediction')
        plt.legend()
        plt.grid()
        fig_name = f"figures/{name.replace(' ', '_').replace('(', '').replace(')', '')}_{target}_Prediction.png"
        plt.savefig(fig_name)
        plt.close()

auto_eval.generate_report()
print('AutoResearch report generated successfully.')
results_df = pd.DataFrame(all_results)
results_df.to_csv('results/ml_results.csv', index=False)
display(results_df)



--- Cible : Arrivals ---
Training Linear Regression for Arrivals...


AutoResearch Insights: Moderate performance. The model captures some dynamics but struggles with high variance. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training Ridge for Arrivals...


AutoResearch Insights: Excellent generalization. The model successfully captures both trend and seasonality without severe overfitting. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training Decision Tree for Arrivals...


AutoResearch Insights: Moderate performance. The model captures some dynamics but struggles with high variance. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training SVM (RBF) for Arrivals...


AutoResearch Insights: Model is worse than baseline average. Temporal degradation or structural break (e.g. COVID) likely broke generalization. Systematic bias detected: The model consistently overpredicts or underpredicts.

--- Cible : Nights ---
Training Linear Regression for Nights...


AutoResearch Insights: Model is worse than baseline average. Temporal degradation or structural break (e.g. COVID) likely broke generalization. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training Ridge for Nights...


AutoResearch Insights: Model is worse than baseline average. Temporal degradation or structural break (e.g. COVID) likely broke generalization. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training Decision Tree for Nights...


AutoResearch Insights: Model is worse than baseline average. Temporal degradation or structural break (e.g. COVID) likely broke generalization. Systematic bias detected: The model consistently overpredicts or underpredicts.
Training SVM (RBF) for Nights...


AutoResearch Insights: Model is worse than baseline average. Temporal degradation or structural break (e.g. COVID) likely broke generalization. Systematic bias detected: The model consistently overpredicts or underpredicts.
AutoResearch report generated successfully.


,Target,Model,Type,R2,RMSE,MAE,MAPE
0,Arrivals,Linear Regression,Machine Learning,0.636154,2.332004e+05,2.030406e+05,0.153462
1,Arrivals,Ridge,Machine Learning,0.779110,1.817014e+05,1.525881e+05,0.116060
2,Arrivals,Decision Tree,Machine Learning,0.693568,2.140117e+05,1.610110e+05,0.103830
3,Arrivals,SVM (RBF),Machine Learning,-4.765202,9.282770e+05,8.437055e+05,0.559944
4,Nights,Linear Regression,Machine Learning,-5.827601,1.582934e+06,1.150433e+06,0.422550
5,Nights,Ridge,Machine Learning,-4.289027,1.393210e+06,1.070005e+06,0.393647
6,Nights,Decision Tree,Machine Learning,-2.368020,1.111773e+06,9.451785e+05,0.353118
7,Nights,SVM (RBF),Machine Learning,-1.517371,9.611746e+05,7.628204e+05,0.268684
